In [1]:
import os
import csv
import glob
import torch
import concurrent.futures
from tqdm import tqdm
from PIL import Image
import easyocr
import cv2

In [ ]:
# --- CONFIG ---
IMAGE_FOLDER = "../XQUANG"
OUTPUT_CSV = r"extracted_texts_data_v1.csv"
CHECKPOINT_FILE = r"checkpoint_v1.txt"
BATCH_SIZE = 64
NUM_WORKERS = 8
SUPPORTED_EXTENSIONS = ("*.png", "*.jpg", "*.jpeg", "*.tiff", "*.bmp")

In [3]:
# --- SETUP ---
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('medium')

In [4]:
def load_processed_files(checkpoint_path):
    processed = set()
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, 'r', encoding='utf-8') as f:
            for line in f:
                processed.add(line.strip())
    return processed

In [5]:
def load_image_np(path):
    """Đọc ảnh thành numpy array."""
    img = cv2.imread(path)
    return img if img is not None else None

In [6]:
print("Initializing EasyOCR with GPU...")

try:
    ocr_model = easyocr.Reader(['vi', 'en'], gpu=True, verbose=False) 
except Exception as e:
    print(f"Error EasyOCR: {e}")

print("\nEasyOCR initialized.")

Initializing EasyOCR with GPU...

EasyOCR initialized.


In [13]:
# 1. Load processed files (checkpoint)
processed_files = load_processed_files(CHECKPOINT_FILE)
print(f"Found {len(processed_files)} processed files from checkpoint.")

Found 160 processed files from checkpoint.


In [8]:
# 2. Get all image paths
all_image_paths = []
for ext in SUPPORTED_EXTENSIONS:
    all_image_paths.extend(
        glob.glob(os.path.join(IMAGE_FOLDER, '**', ext), recursive=True)
    )
print(f"Total found {len(all_image_paths)} images.")

Total found 38681 images.


In [14]:
# 3. Filter unprocessed files
unprocessed_files = [
    path for path in all_image_paths if path not in processed_files
]
if not unprocessed_files:
    print("All files have been processed. Exiting.")
else:
    print(f"Left {len(unprocessed_files)} files need to be processed.")

Left 38521 files need to be processed.


In [17]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [16]:
# 4. Main processing loop
try:
    with open(OUTPUT_CSV, 'a', newline='', encoding='utf-8') as f_csv, open(CHECKPOINT_FILE, 'a', encoding='utf-8') as f_check:
        csv_writer = csv.writer(f_csv)
        if os.path.getsize(OUTPUT_CSV) == 0:
            csv_writer.writerow(['file_path', 'extracted_text'])

        # 5. Process in Batches
        for i in tqdm(range(0, len(unprocessed_files), BATCH_SIZE), desc="Đang xử lý các batch"):
            batch_paths = unprocessed_files[i : i + BATCH_SIZE]
            
            # --- Load ảnh song song ---
            images = {}
            with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
                future_to_path = {executor.submit(load_image_np, p): p for p in batch_paths}
                for future in concurrent.futures.as_completed(future_to_path):
                    path = future_to_path[future]
                    img = future.result()
                    if img is not None:
                        images[path] = img

            if not images:
                continue

            # --- Chạy OCR tuần tự nhưng GPU liên tục ---
            results_to_write = []
            for path, img in images.items():
                try:
                    result = ocr_model.readtext(img, detail=0)
                    full_text = " ".join(result) if result else ""
                    results_to_write.append((path, full_text))
                except Exception as e:
                    results_to_write.append((path, f"ERROR: {e}"))
                    print(f"OCR failed for {path}: {e}")

            # --- Ghi kết quả và checkpoint ---
            for path, text in results_to_write:
                csv_writer.writerow([path, text])
                f_check.write(f"{path}\n")

            f_csv.flush()
            f_check.flush()

            # for path in batch_paths:
            #     try:
            #         result = ocr_model.readtext(path, detail=0)
                        
            #         full_text = " ".join(result) # Nối các text tìm được
            #         results_to_write.append((path, full_text))

            #     except Exception as e:
            #         print(f"\nError {path}: {e}")
            #         results_to_write.append((path, f"ERROR: {e}"))

            # 7. Write results and update checkpoint
            # if results_to_write:
            #     for path, text in results_to_write:
            #         csv_writer.writerow([path, text])
            #         f_check.write(f"{path}\n")
                    
            #     f_csv.flush()
            #     f_check.flush()

except KeyboardInterrupt:
    print("\nProcess interrupted by user. Exiting gracefully.")
except Exception as e:
    print(f"\nUnexpected error: {e}")

print("--- Completed ---")

Đang xử lý các batch:   0%|          | 1/602 [01:20<13:23:56, 80.26s/it]


Process interrupted by user. Exiting gracefully.
--- Completed ---
